# GravBalancer v10.8.2 — LR Sweep Benchmark
# Supports: toy_2d (mode coverage), cifar10 (FID/KID), afhq (FID/KID)
# Runs all modes × LR grid, saves combined report + GIF animations

In [1]:
# ════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════

DATASET = "cifar10"          # "toy_2d" | "cifar10" | "afhq"

# toy_2d specific
GRID_SIZE = 10
GRID_SPACING = 2.0
GRID_STD = 0.05
N_DATA = 500_000

# Image specific
IMG_SIZE = 32               # 32 for cifar10, 64 for afhq recommended
IMG_CHANNELS = 3

# Training
EPOCHS = 300
LR_GRID = [5e-3, 1e-2]
LR_G = LR_GRID[0]
LR_D = LR_GRID[0]
BATCH_SIZE = 512

# Z_DIM: 2 for toy_2d (interpretable), 128 for images (DCGAN standard)
Z_DIM_TOY = 2
Z_DIM_IMG = 128
Z_DIM = Z_DIM_TOY if DATASET == "toy_2d" else Z_DIM_IMG

SEED = 42
MODES = ["baseline", "grav", "div", "grav_div"]

# GravBalancer
GRAV_WARMUP_EPOCHS = 1
GRAV_VERSION = "v10.8"      # GravBalancer version tag
SWEEP_VERSION = "10.8.2"    # this notebook version

# DiversitySentinel
DIV_WARMUP_EPOCHS = 3
DIV_LAMBDA_MAX = 20.0
DIV_PID_TARGET_RATIO = 0.10

# GIF
GIF_ENABLED = True
GIF_SAMPLE_EVERY_N_EPOCHS = 5
GIF_FPS = 4
GIF_DPI = 100
GIF_N_SAMPLES_TOY = 3000       # scatter points per frame (toy_2d)
GIF_GRID_COLS = 8              # image grid layout (cifar/afhq)
GIF_GRID_ROWS = 8
GIF_N_SAMPLES_IMG = GIF_GRID_COLS * GIF_GRID_ROWS  # 64

# Image quality metrics
COMPUTE_FID = True
COMPUTE_KID = True
FID_N_SAMPLES = 2048

# Output
OUTPUT_DIR = "benchmark_output"
SAVE_JSON = True

In [2]:
# ════════════════════════════════════════════════════════════════════
# Imports & Setup
# ════════════════════════════════════════════════════════════════════

import os, sys, time, io, json, importlib, math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from grav_balancer import GravBalancer
from diversity_sentinel import DiversitySentinel, DiversitySentinelConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GravBalancer: {GRAV_VERSION}, Dataset: {DATASET}, Z_DIM: {Z_DIM}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

IS_TOY = (DATASET == "toy_2d")
IS_IMG = not IS_TOY

def seed_everything(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def get_mode_centers(grid_size, spacing):
    offset = (grid_size - 1) / 2.0 * spacing
    return [(i * spacing - offset, j * spacing - offset)
            for i in range(grid_size) for j in range(grid_size)]

Device: cuda
GravBalancer: v10_8, Dataset: cifar10, Z_DIM: 128


In [3]:
# ═══ Seed & Data ═══

def load_dataset():
    if DATASET == "toy_2d":
        centers = get_mode_centers(GRID_SIZE, GRID_SPACING)
        n_modes = len(centers)
        n_per = N_DATA // n_modes
        pts = []
        for cx, cy in centers:
            pts.append(np.column_stack([
                np.random.normal(cx, GRID_STD, n_per),
                np.random.normal(cy, GRID_STD, n_per)]))
        data = np.vstack(pts).astype(np.float32)
        np.random.shuffle(data)
        tensor = torch.from_numpy(data)
        loader = DataLoader(TensorDataset(tensor), batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True)
        return tensor, loader, {"type": "toy_2d", "n_modes": n_modes, "grid_size": GRID_SIZE}

    elif DATASET == "cifar10":
        from torchvision import datasets, transforms
        tf = transforms.Compose([
            transforms.Resize(IMG_SIZE), transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)])
        ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
        loader = DataLoader(ds, batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True, num_workers=0)
        return None, loader, {"type": "cifar10", "img_size": IMG_SIZE, "channels": 3}

    elif DATASET == "afhq":
        from torchvision import datasets, transforms
        afhq_root = './data/afhq/train'
        if not os.path.isdir(afhq_root):
            raise FileNotFoundError(
                f"AFHQ not found at {afhq_root}.\n"
                f"Download: https://github.com/clovaai/stargan-v2#animal-faces-hq-dataset-afhq\n"
                f"Expected: {afhq_root}/cat/, {afhq_root}/dog/, {afhq_root}/wild/")
        tf = transforms.Compose([
            transforms.Resize(IMG_SIZE + IMG_SIZE // 8),
            transforms.CenterCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)])
        ds = datasets.ImageFolder(root=afhq_root, transform=tf)
        loader = DataLoader(ds, batch_size=BATCH_SIZE,
                            shuffle=True, drop_last=True, num_workers=0)
        print(f"  AFHQ: {len(ds)} images, {len(ds.classes)} classes: {ds.classes}")
        return None, loader, {"type": "afhq", "img_size": IMG_SIZE, "channels": 3,
                              "n_classes": len(ds.classes)}
    else:
        raise ValueError(f"Unknown dataset: {DATASET}")

seed_everything(SEED)
data_tensor, data_loader, data_meta = load_dataset()
steps_per_epoch = len(data_loader)
print(f"Steps/epoch: {steps_per_epoch}, Meta: {data_meta}")

Files already downloaded and verified
Steps/epoch: 97, Meta: {'type': 'cifar10', 'img_size': 32, 'channels': 3}


In [4]:
# ═══ Models ═══

class ToyGenerator(nn.Module):
    def __init__(self, z_dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2))
    def forward(self, z): return self.net(z)

class ToyDiscriminator(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x)

class ImageGenerator(nn.Module):
    def __init__(self, z_dim=128, img_size=32, channels=3, ngf=64):
        super().__init__()
        assert img_size in (32, 64)
        layers = [
            nn.ConvTranspose2d(z_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True)]
        if img_size == 64:
            layers += [
                nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
                nn.BatchNorm2d(ngf), nn.ReLU(True),
                nn.ConvTranspose2d(ngf, channels, 4, 2, 1, bias=False)]
        else:
            layers += [nn.ConvTranspose2d(ngf*2, channels, 4, 2, 1, bias=False)]
        layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)
    def forward(self, z): return self.net(z.view(z.size(0), -1, 1, 1))

class ImageDiscriminator(nn.Module):
    def __init__(self, img_size=32, channels=3, ndf=64):
        super().__init__()
        assert img_size in (32, 64)
        layers = [
            nn.Conv2d(channels, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True)]
        if img_size == 64:
            layers += [
                nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
                nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, True),
                nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False)]
        else:
            layers += [nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).view(-1, 1)

def make_models():
    if IS_TOY:
        return ToyGenerator(z_dim=Z_DIM).to(device), ToyDiscriminator().to(device)
    else:
        ch = data_meta.get("channels", 3)
        return (ImageGenerator(z_dim=Z_DIM, img_size=IMG_SIZE, channels=ch).to(device),
                ImageDiscriminator(img_size=IMG_SIZE, channels=ch).to(device))

In [5]:
# ═══ Metrics ═══

def count_covered_modes(fake_pts, grid_size=GRID_SIZE, spacing=GRID_SPACING,
                        radius=0.5, min_count=5):
    centers = get_mode_centers(grid_size, spacing)
    n_modes = len(centers)
    counts = np.zeros(n_modes)
    for k, (cx, cy) in enumerate(centers):
        dists = np.sqrt((fake_pts[:, 0] - cx)**2 + (fake_pts[:, 1] - cy)**2)
        counts[k] = np.sum(dists < radius)
    covered = int(np.sum(counts >= min_count))
    active = counts[counts >= min_count]
    cv = float(np.std(active) / (np.mean(active) + 1e-9)) if len(active) > 0 else 999.
    return {'covered': covered, 'total': n_modes, 'pct': covered / n_modes * 100,
            'uniformity_cv': cv, 'per_mode_counts': counts.tolist()}

def compute_wasserstein1_toy(fake_pts, grid_size=GRID_SIZE, spacing=GRID_SPACING):
    centers_arr = np.array(get_mode_centers(grid_size, spacing))
    all_mins = []
    for i in range(0, fake_pts.shape[0], 5000):
        d = np.linalg.norm(fake_pts[i:i+5000, None, :] - centers_arr[None, :, :], axis=2)
        all_mins.append(np.min(d, axis=1))
    return float(np.mean(np.concatenate(all_mins)))

def compute_mode_kl_toy(fake_pts, grid_size=GRID_SIZE, spacing=GRID_SPACING, radius=0.5, eps=1e-8):
    centers = get_mode_centers(grid_size, spacing)
    n_modes = len(centers)
    counts = np.zeros(n_modes)
    for k, (cx, cy) in enumerate(centers):
        dists = np.sqrt((fake_pts[:, 0] - cx)**2 + (fake_pts[:, 1] - cy)**2)
        counts[k] = np.sum(dists < radius)
    q = np.clip(counts / (counts.sum() + eps), eps, None)
    p = np.ones(n_modes) / n_modes
    return float(np.sum(p * np.log(p / q)))

def _generate_fake_uint8(G, n_samples, batch_size=256):
    G.eval()
    fakes = []
    with torch.no_grad():
        for i in range(0, n_samples, batch_size):
            bs = min(batch_size, n_samples - i)
            z = torch.randn(bs, Z_DIM, device=device)
            imgs = G(z).clamp(-1, 1)
            fakes.append(((imgs + 1) * 127.5).to(torch.uint8).cpu())
    G.train()
    return torch.cat(fakes)[:n_samples]

def _collect_real_uint8(loader, n_samples):
    reals = []
    total = 0
    for batch in loader:
        imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
        reals.append(((imgs.clamp(-1, 1) + 1) * 127.5).to(torch.uint8))
        total += imgs.size(0)
        if total >= n_samples: break
    return torch.cat(reals)[:n_samples]

def try_compute_fid_kid(G, n_samples=2048):
    if IS_TOY: return {}
    metrics = {}
    try:
        from torchmetrics.image.fid import FrechetInceptionDistance
        from torchmetrics.image.kid import KernelInceptionDistance
    except ImportError:
        print("  [!] torchmetrics not available -- pip install torchmetrics[image]")
        return {}
    try:
        fakes = _generate_fake_uint8(G, n_samples)
        reals = _collect_real_uint8(data_loader, n_samples)
        if fakes.shape[1] == 1:
            fakes = fakes.repeat(1, 3, 1, 1)
            reals = reals.repeat(1, 3, 1, 1)

        # Feed in batches to avoid giant 299×299 resize allocation
        FID_BATCH = 256
        if COMPUTE_FID:
            fid = FrechetInceptionDistance(normalize=False)
            for i in range(0, len(reals), FID_BATCH):
                fid.update(reals[i:i+FID_BATCH], real=True)
            for i in range(0, len(fakes), FID_BATCH):
                fid.update(fakes[i:i+FID_BATCH], real=False)
            metrics['fid'] = float(fid.compute())
            del fid

        if COMPUTE_KID:
            subset = min(1000, n_samples)
            kid = KernelInceptionDistance(subset_size=subset, normalize=False)
            for i in range(0, len(reals), FID_BATCH):
                kid.update(reals[i:i+FID_BATCH], real=True)
            for i in range(0, len(fakes), FID_BATCH):
                kid.update(fakes[i:i+FID_BATCH], real=False)
            km, ks = kid.compute()
            metrics['kid_mean'] = float(km); metrics['kid_std'] = float(ks)
            del kid

        del fakes, reals
    except Exception as e:
        print(f"  [!] FID/KID failed: {e}")
    finally:
        import gc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metrics

def compute_all_metrics(G, fake_pts_np=None):
    metrics = {}
    if IS_TOY:
        cov = count_covered_modes(fake_pts_np)
        metrics.update({
            'mode_coverage': cov['pct'] / 100.0,
            'n_modes_covered': cov['covered'], 'n_modes_total': cov['total'],
            'uniformity_cv': cov['uniformity_cv'],
            'per_mode_counts': cov['per_mode_counts'],
            'wasserstein1': compute_wasserstein1_toy(fake_pts_np),
            'mode_kl': compute_mode_kl_toy(fake_pts_np)})
    else:
        metrics.update(try_compute_fid_kid(G, FID_N_SAMPLES))
    return metrics

In [6]:
# ═══ DiversitySentinel factory + GIF helpers ═══

def make_sentinel(device='cpu', warmup_steps=200,
                  lambda_max=20.0, pid_target_ratio=0.10):
    if IS_TOY:
        proj_dim, n_proj = 2, 50
    else:
        proj_dim, n_proj = 128, 128
    cfg = DiversitySentinelConfig(
        lambda_max=lambda_max, pid_target_ratio=pid_target_ratio,
        warmup_steps=warmup_steps, projector_dim=proj_dim,
        n_projections=n_proj, respect_grav_calm=True)
    return DiversitySentinel(cfg, device=device)

def make_image_grid(images_tensor, nrow=GIF_GRID_COLS):
    imgs = (images_tensor.clamp(-1, 1) + 1) / 2.0
    n, c, h, w = imgs.shape
    ncol = nrow
    nrow_actual = min(math.ceil(n / ncol), GIF_GRID_ROWS)
    n_show = nrow_actual * ncol
    pad = 2
    canvas = torch.ones(c, nrow_actual * h + (nrow_actual - 1) * pad,
                        ncol * w + (ncol - 1) * pad)
    for idx in range(min(n_show, n)):
        r, col = idx // ncol, idx % ncol
        y, x = r * (h + pad), col * (w + pad)
        canvas[:, y:y+h, x:x+w] = imgs[idx]
    arr = (canvas.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    if c == 1: arr = arr.squeeze(-1)
    return Image.fromarray(arr)

def capture_gif_frame_toy(G, fixed_z, data_tensor, mode, epoch, epochs):
    G.eval()
    with torch.no_grad():
        snap = G(fixed_z).cpu().numpy()
    G.train()
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    if data_tensor is not None:
        ax.scatter(data_tensor[:3000, 0].numpy(), data_tensor[:3000, 1].numpy(),
                   s=1, alpha=0.1, c='steelblue')
    ax.scatter(snap[:, 0], snap[:, 1], s=2, alpha=0.5, c='orangered')
    lim = (GRID_SIZE - 1) / 2.0 * GRID_SPACING + 1.5
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_title(f'{mode}  epoch {epoch+1}/{epochs}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.15)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=GIF_DPI, bbox_inches='tight')
    buf.seek(0); frame = Image.open(buf).copy(); plt.close(fig)
    return frame

def capture_gif_frame_img(G, fixed_z, mode, epoch, epochs):
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z).cpu()
    G.train()
    grid_img = make_image_grid(imgs, nrow=GIF_GRID_COLS)
    from PIL import ImageDraw
    title_h = 24
    full = Image.new('RGB', (grid_img.width, grid_img.height + title_h), (255, 255, 255))
    full.paste(grid_img, (0, title_h))
    draw = ImageDraw.Draw(full)
    draw.text((4, 4), f'{mode}  epoch {epoch+1}/{epochs}', fill=(0, 0, 0))
    return full

In [7]:
# ═══ Training Loop (v10.8 diagnostics) ═══

def train_gan(mode='grav_div'):
    seed_everything(SEED)
    use_grav = 'grav' in mode
    use_div = 'div' in mode

    G, D = make_models()
    opt_G = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.0, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.999))

    balancer = None
    if use_grav:
        ref_lr = (LR_G + LR_D) / 2
        balancer = GravBalancer(
            n_players=2, base_lr=ref_lr,
            warmup_steps=GRAV_WARMUP_EPOCHS * steps_per_epoch)

    sentinel = None
    if use_div:
        sentinel = make_sentinel(
            device=str(device),
            warmup_steps=DIV_WARMUP_EPOCHS * steps_per_epoch,
            lambda_max=DIV_LAMBDA_MAX,
            pid_target_ratio=DIV_PID_TARGET_RATIO)

    torch.manual_seed(SEED)
    gif_n = GIF_N_SAMPLES_TOY if IS_TOY else GIF_N_SAMPLES_IMG
    fixed_z = torch.randn(gif_n, Z_DIM).to(device)

    grav_keys = [
        'calm', 'shock_ema', 'comfort_factor',
        'wall_ema_fast', 'comfort_active', 'wall_any',
        'headroom_min', 'i_enabled', 'i_sat', 'gaps_db_nz', 'noise_norm',
        'authority_factor', 'panic_factor',
        'panic_active', 'panic_symptom_score', 'panic_shock_stress', 'panic_shock_stress_ema',
        'noise_ref', 'u_cap_eff', 'calm_cause', 'trip_active', 'control_stress',
        'stress_auth_factor', 'distress_level', 'shock_path', 'shock_thr', 'shock_scale',
        'shock_trip_trigger', 'panic_symptom_thr_eff',
        'auth_min_eff', 'intervention_rate', 'health_proxy',
        'dyn_deadband_z_eff', 'world_drift', 'idle_fraction',
        'climate_E', 'climate_E_fast', 'climate_E_slow',
        'climate_lr_meta', 'climate_hold_applied', 'climate_hold_next',
        'climate_lr_eff_mean']

    grav_defaults = {
        'calm': 0, 'shock_ema': 0, 'comfort_factor': 1.0,
        'wall_ema_fast': 0, 'comfort_active': 0, 'wall_any': 0,
        'headroom_min': 1.0, 'i_enabled': 0, 'i_sat': 0,
        'gaps_db_nz': 0, 'noise_norm': 0,
        'authority_factor': 1.0, 'panic_factor': 1.0,
        'panic_active': 0, 'panic_symptom_score': 0.0,
        'panic_shock_stress': 0.0, 'panic_shock_stress_ema': 0.0,
        'noise_ref': 0.0, 'u_cap_eff': 0.0, 'calm_cause': 0, 'trip_active': 0,
        'control_stress': 0.0, 'stress_auth_factor': 1.0, 'distress_level': 0,
        'shock_path': 0, 'shock_thr': 0.0, 'shock_scale': 0.0, 'shock_trip_trigger': 0,
        'panic_symptom_thr_eff': 0.0,
        'auth_min_eff': 0.0, 'intervention_rate': 0.0, 'health_proxy': 0.0,
        'dyn_deadband_z_eff': 0.0, 'world_drift': 0.0, 'idle_fraction': 0.0,
        'climate_E': 0.0, 'climate_E_fast': 0.0, 'climate_E_slow': 0.0,
        'climate_lr_meta': 1.0, 'climate_hold_applied': 0, 'climate_hold_next': 0,
        'climate_lr_eff_mean': 0.0}

    hist = {k: [] for k in [
        'loss_D', 'loss_G', 'lr_D', 'lr_G',
        'div_lambda', 'div_lambda_raw', 'div_lambda_eff', 'div_loss', 'div_spread']
        + [f'grav_{k}' for k in grav_keys]}

    gif_frames = []
    curr_lr_d, curr_lr_g = LR_D, LR_G
    t_start = time.time()

    for epoch in range(EPOCHS):
        for step, batch in enumerate(data_loader):
            real = batch[0].to(device) if isinstance(batch, (list, tuple)) else batch.to(device)
            bs = real.size(0)

            opt_D.zero_grad()
            z = torch.randn(bs, Z_DIM, device=device)
            fake = G(z)
            d_real = D(real)
            d_fake_det = D(fake.detach())
            loss_D = F.relu(1.0 - d_real).mean() + F.relu(1.0 + d_fake_det).mean()
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad()
            d_fake_G = D(fake)
            loss_G_main = -d_fake_G.mean()
            loss_G_total = loss_G_main
            div_info = {}
            if use_div:
                loss_div, div_info = sentinel.compute(fake, real)
                loss_G_total = loss_G_main + loss_div
            loss_G_total.backward(); opt_G.step()

            if use_div:
                sentinel.step(loss_G_main.item(), bs)
            if use_grav:
                with torch.no_grad():
                    # Softplus proxy (always >= 0, no dead zones)
                    proxy_d = (F.softplus(-d_real).mean() + F.softplus(d_fake_det).mean()).item()
                    proxy_g = F.softplus(-d_fake_G).mean().item()
                    lrs = balancer.adjust(proxy_d, proxy_g)
                    curr_lr_d, curr_lr_g = lrs[0], lrs[1]
                for pg in opt_D.param_groups: pg['lr'] = curr_lr_d
                for pg in opt_G.param_groups: pg['lr'] = curr_lr_g
                if use_div:
                    sentinel.notify_grav_state(balancer.last_metrics)

            hist['loss_D'].append(loss_D.item())
            hist['loss_G'].append(loss_G_main.item())
            hist['lr_D'].append(curr_lr_d); hist['lr_G'].append(curr_lr_g)
            hist['div_lambda'].append(div_info.get('div/lambda', 0))
            hist['div_lambda_raw'].append(div_info.get('div/lambda_raw_pid', 0))
            hist['div_loss'].append(div_info.get('div/loss_total', 0))
            hist['div_spread'].append(div_info.get('div/spread_fake_batch', 0))
            hist['div_lambda_eff'].append(div_info.get('div/lambda_effective', 0))
            gm = balancer.last_metrics if use_grav else {}
            for key in grav_keys:
                hist[f'grav_{key}'].append(float(gm.get(key, grav_defaults[key])))

        if GIF_ENABLED and ((epoch + 1) % GIF_SAMPLE_EVERY_N_EPOCHS == 0 or epoch == 0):
            if IS_TOY:
                frame = capture_gif_frame_toy(G, fixed_z, data_tensor, mode, epoch, EPOCHS)
            else:
                frame = capture_gif_frame_img(G, fixed_z, mode, epoch, EPOCHS)
            gif_frames.append(frame)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            lam_s = f' lam={hist["div_lambda"][-1]:.2f}' if use_div else ''
            if use_grav and not gm.get('warmup', False):
                from collections import Counter
                calm_pct = np.mean(hist["grav_calm"][-steps_per_epoch:])*100
                trip_pct = np.mean(hist["grav_trip_active"][-steps_per_epoch:])*100
                causes_ep = [int(c) for c in hist["grav_calm_cause"][-steps_per_epoch:] if c > 0]
                cc = Counter(causes_ep)
                cause_s = ' '.join(f'c{k}={v}' for k,v in sorted(cc.items())) if cc else 'none'
                calm_s = f' calm={calm_pct:.0f}% trip={trip_pct:.0f}% [{cause_s}]'
                auth_s = f' auth={np.mean(hist["grav_authority_factor"][-steps_per_epoch:]):.3f}'
                stress_s = f' stress={np.mean(hist["grav_control_stress"][-steps_per_epoch:]):.3f}'
                db_eff = np.mean(hist["grav_dyn_deadband_z_eff"][-steps_per_epoch:])
                idle_f = np.mean(hist["grav_idle_fraction"][-steps_per_epoch:])
                idle_s = f' db={db_eff:.2f} idle={idle_f:.0%}'
                pf = np.mean(hist["grav_panic_factor"][-steps_per_epoch:])
                panic_s = f' panic={pf:.3f}' if pf < 1.0 - 1e-4 else ''
                clm_meta = np.mean(hist["grav_climate_lr_meta"][-steps_per_epoch:])
                clm_hold = np.mean(hist["grav_climate_hold_applied"][-steps_per_epoch:])*100
                climate_s = f' lr_meta={clm_meta:.3f} hold={clm_hold:.0f}%' if clm_meta < 1.0 - 1e-4 or clm_hold > 0.5 else ''
            elif use_grav:
                calm_s = ' [warmup]'; auth_s = ''; panic_s = ''; stress_s = ''; idle_s = ''; climate_s = ''
            else:
                calm_s = ''; auth_s = ''; panic_s = ''; stress_s = ''; idle_s = ''; climate_s = ''
            print(f'  [{mode}] ep {epoch+1:3d}/{EPOCHS}  '
                  f'D={np.mean(hist["loss_D"][-steps_per_epoch:]):.3f}  '
                  f'G={np.mean(hist["loss_G"][-steps_per_epoch:]):.3f}{lam_s}{calm_s}{auth_s}{stress_s}{panic_s}{idle_s}{climate_s}')

    train_time = time.time() - t_start

    # ── Free training objects no longer needed ──
    del D, opt_D, opt_G
    if balancer is not None:
        balancer_out = balancer  # keep for report
    else:
        balancer_out = None
    del sentinel
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    G.eval()
    with torch.no_grad():
        if IS_TOY:
            final_z = torch.randn(GIF_N_SAMPLES_TOY, Z_DIM, device=device)
            final_fake = G(final_z).cpu().numpy()
        else:
            final_fake = None
    final_real = data_tensor[:GIF_N_SAMPLES_TOY].numpy() if IS_TOY and data_tensor is not None else None

    gif_path = None
    if GIF_ENABLED and len(gif_frames) > 1:
        lr_tag = f'{LR_G:.0e}'.replace('+', '').replace('-0', '-')
        gif_path = os.path.join(OUTPUT_DIR, f'evolution_lr{lr_tag}_{mode}.gif')
        gif_frames[0].save(gif_path, save_all=True, append_images=gif_frames[1:],
                           duration=int(1000 / GIF_FPS), loop=0)
        print(f'  GIF: {gif_path} ({len(gif_frames)} frames)')

    # Free GIF frames memory
    del gif_frames
    gif_frames = []
    import gc; gc.collect()

    return {'hist': hist, 'fake': final_fake, 'real': final_real,
            'G': G, 'gif_frames': gif_frames, 'gif_path': gif_path,
            'train_time': train_time, 'balancer': balancer_out}

In [8]:
# ════════════════════════════════════════════════════════════════════
# LR SWEEP — Run all modes for each LR, save combined report
# ════════════════════════════════════════════════════════════════════

import io, time
from collections import Counter

def generate_report(results, all_metrics, config_vars):
    buf = io.StringIO()
    w = buf.write

    # ── Config ──
    w(f"{'═' * 80}\n")
    w(f"  BENCHMARK REPORT — GravBalancer {config_vars.get('grav_version', '?')}\n")
    w(f"{'═' * 80}\n\n")
    w(f"Config:\n")
    for k, v in sorted(config_vars.items()):
        w(f"  {k:30s} = {v}\n")

    # ── Summary table ──
    w(f"\n{'═' * 80}\n")
    w(f"  SUMMARY TABLE\n")
    w(f"{'═' * 80}\n")
    if IS_IMG:
        header = (f"{'Mode':>14s} {'FID':>10s} {'KID_m':>10s} {'KID_s':>10s}"
                  f" {'D_final':>10s} {'G_final':>10s}"
                  f" {'calm%':>7s} {'trip%':>7s} {'auth':>7s} {'stress':>7s} {'panic%':>7s}\n")
    else:
        header = (f"{'Mode':>14s} {'modes':>8s} {'CV':>8s} {'W1':>8s} {'KL':>10s}"
              f" {'D_final':>10s} {'G_final':>10s}"
              f" {'calm%':>7s} {'trip%':>7s} {'auth':>7s} {'stress':>7s} {'panic%':>7s}\n")
    w(header)
    w(f"{'-' * 120}\n")
    for mode, m in all_metrics.items():
        if mode == '_config':
            continue
        if IS_IMG:
            fid_v = f"{m.get('fid', -1):.1f}" if 'fid' in m else "--"
            kid_m = f"{m.get('kid_mean', -1):.4f}" if 'kid_mean' in m else "--"
            kid_s = f"{m.get('kid_std', -1):.4f}" if 'kid_std' in m else "--"
        mc = str(m.get('n_modes_covered', '?'))
        mt = str(m.get('n_modes_total', '?'))
        cv = f"{m.get('uniformity_cv', 0):.3f}"
        w1 = f"{m.get('wasserstein1', 0):.3f}"
        kl = f"{m.get('mode_kl', 0):.4f}"
        df = f"{m.get('loss_D_final', 0):.2f}"
        gf = f"{m.get('loss_G_final', 0):.2f}"

        is_grav = 'grav' in mode
        h = results[mode]["hist"] if mode in results else {}
        # Calm rate from history (more accurate than tail-only metric)
        calm_arr = h.get("grav_calm", [])
        trip_arr = h.get("grav_trip_active", [])
        calm_s = f"{np.mean(calm_arr)*100:.1f}%" if calm_arr and is_grav else "—"
        trip_s = f"{np.mean(trip_arr)*100:.1f}%" if trip_arr and is_grav else "—"
        auth_s = f"{m.get('grav_authority_factor_mean', 0):.3f}" if is_grav else "—"
        # Control stress from history
        stress_arr = h.get("grav_control_stress", [])
        stress_s = f"{np.mean(stress_arr):.3f}" if stress_arr and is_grav else "—"
        panic_arr = h.get("grav_panic_active", [])
        panic_s = f"{np.mean(panic_arr)*100:.1f}%" if panic_arr and is_grav else "—"

        if IS_IMG:
            w(f"{mode:>14s} {fid_v:>10s} {kid_m:>10s} {kid_s:>10s}"
              f" {df:>10s} {gf:>10s}"
              f" {calm_s:>7s} {trip_s:>7s} {auth_s:>7s} {stress_s:>7s} {panic_s:>7s}\n")
            continue
        w(f"{mode:>14s} {mc:>4s}/{mt:<3s} {cv:>8s} {w1:>8s} {kl:>10s}"
          f" {df:>10s} {gf:>10s}"
          f" {calm_s:>7s} {trip_s:>7s} {auth_s:>7s} {stress_s:>7s} {panic_s:>7s}\n")

    # ── Per-mode detailed diagnostics ──
    for mode in ["grav", "grav_div"]:
        if mode not in results:
            continue
        h = results[mode]["hist"]
        gb = results[mode].get("balancer")
        total = len(h["grav_calm"])

        w(f"\n{'═' * 80}\n")
        w(f"  {mode} — Detailed Diagnostics  ({total} steps)\n")
        w(f"{'═' * 80}\n")

        # Calm
        causes = [int(c) for c in h["grav_calm_cause"] if c > 0]
        cc = Counter(causes)
        calm_total = sum(1 for c in h["grav_calm"] if c > 0)
        cause_names = {1: "shock(BYPASS)", 2: "mech", 3: "legacy", 4: "preempt"}
        w(f"\n  Calm: {calm_total}/{total} ({calm_total/total:.1%})\n")
        for k in sorted(cc):
            w(f"    cause={k} ({cause_names.get(k, '?')}): {cc[k]} ({cc[k]/total:.1%})\n")
        if not cc:
            w(f"    (none)\n")

        # Tripwire
        trips = h.get("grav_trip_active", [])
        trip_total = int(sum(trips)) if trips else 0
        w(f"\n  Tripwire: {trip_total}/{total} ({trip_total/total:.1%})\n")

        # Shock paths
        paths = [int(p) for p in h["grav_shock_path"]]
        pp = Counter(paths)
        path_names = {0: "NONE", 1: "BYPASS->calm", 2: "SUSTAINED->trip", 3: "NORMAL->trip"}
        w(f"\n  Shock paths:\n")
        for k in sorted(pp):
            pct = pp[k] / total
            bar = '#' * int(pct * 40)
            w(f"    path={k} ({path_names.get(k, '?'):>16s}): {pp[k]:>6d} ({pct:6.1%}) {bar}\n")

        # Shock internals
        shock_ema = np.array(h["grav_shock_ema"])
        shock_thr = np.array(h["grav_shock_thr"])
        shock_scale = np.array(h.get("grav_shock_scale", [0.0]*total))
        noise_ref = np.array(h["grav_noise_ref"])
        noise_norm = np.array(h["grav_noise_norm"])
        n = min(steps_per_epoch, total)
        w(f"\n  Shock internals (last epoch / full run):\n")
        w(f"{'':>20s} {'last epoch':>14s} {'full run':>14s}\n")
        w(f"    shock_ema:    {np.mean(shock_ema[-n:]):>14.6f} {np.mean(shock_ema):>14.6f}\n")
        w(f"    shock_thr:    {np.mean(shock_thr[-n:]):>14.6f} {np.mean(shock_thr):>14.6f}\n")
        w(f"    shock_scale:  {np.mean(shock_scale[-n:]):>14.6f} {np.mean(shock_scale):>14.6f}\n")
        w(f"    noise_ref:    {np.mean(noise_ref[-n:]):>14.6f} {np.mean(noise_ref):>14.6f}\n")
        w(f"    noise_norm:   {np.mean(noise_norm[-n:]):>14.4f} {np.mean(noise_norm):>14.4f}\n")
        w(f"    shock_ema max:{np.max(shock_ema[-n:]):>14.6f} {np.max(shock_ema):>14.6f}\n")
        headroom = shock_thr[-n:] - shock_ema[-n:]
        w(f"    headroom:     mean={np.mean(headroom):.6f}  min={np.min(headroom):.6f}\n")

        # Balancer state
        if gb is not None:
            warmup_thr = gb._shock_k * gb._shock_snap
            live_thr = gb._shock_thr
            w(f"\n  Balancer final state:\n")
            w(f"    _shock_snap     = {gb._shock_snap:.6f}  (warmup, diagnostic only)\n")
            w(f"    _snap_vol_thr   = {gb._snap_vol_thr:.6f}\n")
            w(f"    _shock_thr      = {gb._shock_thr:.6f}  (live operational)\n")
            w(f"    _shock_scale    = {gb._shock_scale:.6f}\n")
            w(f"    shock_k         = {gb._shock_k:.1f}\n")
            w(f"    noise_ref       = {gb._noise_ref:.6f}\n")
            w(f"    shock_ema       = {gb._shock_ema:.6f}\n")
            w(f"    rel_vol_ema     = {gb._rel_vol_ema:.6f}\n")
            w(f"    warmup_thr      = {warmup_thr:.6f}  (what v10.7.2 would have used)\n")
            w(f"    live_thr        = {live_thr:.6f}  (v10.7.3 operational)\n")
            if warmup_thr > 0:
                w(f"    thr ratio       = {live_thr / warmup_thr:.2f}x\n")
            # v10.7.4 state
            w(f"    dyn_deadband_eff = {gb._dyn_deadband_z_eff:.4f}\n")
            w(f"    intervention_rate= {gb._intervention_rate_ema:.4f}\n")
            w(f"    auth_min_eff     = {gb._auth_min_eff:.4f}\n")

        # Control stress
        stress = np.array(h.get("grav_control_stress", []))
        stress_auth = np.array(h.get("grav_stress_auth_factor", []))
        if len(stress) > 0:
            w(f"\n  Control stress:\n")
            w(f"    stress:       mean={np.mean(stress):.4f}  max={np.max(stress):.4f}\n")
            w(f"    stress_auth:  mean={np.mean(stress_auth):.4f}  min={np.min(stress_auth):.4f}\n")

        # Authority
        auth = np.array(h["grav_authority_factor"])
        w(f"\n  Authority:\n")
        w(f"    mean={np.mean(auth):.4f}  min={np.min(auth):.4f}  max={np.max(auth):.4f}\n")

        # v10.7.4: anti-iatrogenic + dynamic idle
        auth_min_eff = np.array(h.get("grav_auth_min_eff", []))
        interv_rate = np.array(h.get("grav_intervention_rate", []))
        health_proxy = np.array(h.get("grav_health_proxy", []))
        db_eff = np.array(h.get("grav_dyn_deadband_z_eff", []))
        w_drift = np.array(h.get("grav_world_drift", []))
        idle_frac = np.array(h.get("grav_idle_fraction", []))
        if len(auth_min_eff) > 0:
            w(f"\n  Dynamic idle:\n")
            w(f"{'':>20s} {'last epoch':>14s} {'full run':>14s}\n")
            w(f"    auth_min_eff: {np.mean(auth_min_eff[-n:]):>14.4f} {np.mean(auth_min_eff):>14.4f}\n")
            w(f"    interv_rate:  {np.mean(interv_rate[-n:]):>14.4f} {np.mean(interv_rate):>14.4f}\n")
            w(f"    health_proxy: {np.mean(health_proxy[-n:]):>14.4f} {np.mean(health_proxy):>14.4f}\n")
            w(f"    db_eff:       {np.mean(db_eff[-n:]):>14.3f} {np.mean(db_eff):>14.3f}\n")
            w(f"    world_drift:  {np.mean(w_drift[-n:]):>14.3f} {np.mean(w_drift):>14.3f}\n")
            w(f"    idle_frac:    {np.mean(idle_frac[-n:]):>14.3f} {np.mean(idle_frac):>14.3f}\n")

        # Panic
        panic_active = np.array(h["grav_panic_active"])
        panic_score = np.array(h["grav_panic_symptom_score"])
        panic_shock_stress = np.array(h["grav_panic_shock_stress"])
        panic_factor = np.array(h["grav_panic_factor"])
        panic_total = int(np.sum(panic_active))
        w(f"\n  Panic: {panic_total}/{total} ({panic_total/total:.1%})\n")
        w(f"    symptom_score: mean={np.mean(panic_score):.4f}  max={np.max(panic_score):.4f}\n")
        w(f"    shock_stress:  mean={np.mean(panic_shock_stress):.4f}  max={np.max(panic_shock_stress):.4f}\n")
        panic_shock_stress_ema = np.array(h["grav_panic_shock_stress_ema"])
        w(f"    shock_stress_ema: mean={np.mean(panic_shock_stress_ema):.4f}  max={np.max(panic_shock_stress_ema):.4f}\n")
        thr_eff_arr = np.array(h.get("grav_panic_symptom_thr_eff", [0]))
        if len(thr_eff_arr) > 0:
            w(f"    symptom_thr_eff: {thr_eff_arr[-1]:.4f}\n")
        if panic_total > 0:
            w(f"    panic_factor:  mean={np.mean(panic_factor):.4f}  min={np.min(panic_factor):.4f}\n")

        # Wall/Hold/I
        w(f"\n  Wall/Hold/I-term:\n")
        w(f"    wall_any:     {np.mean(h.get('grav_wall_any', [0])):.4f}\n")
        w(f"    i_enabled:    {np.mean(h.get('grav_i_enabled', [0])):.4f}\n")
        w(f"    i_sat:        {np.mean(h.get('grav_i_sat', [0])):.6f}\n")

        # v10.8 Climate control
        clm_E_slow = np.array(h.get("grav_climate_E_slow", []))
        clm_lr_meta = np.array(h.get("grav_climate_lr_meta", []))
        clm_hold = np.array(h.get("grav_climate_hold_applied", []))
        clm_eff = np.array(h.get("grav_climate_lr_eff_mean", []))
        if len(clm_lr_meta) > 0 and np.any(clm_lr_meta < 1.0 - 1e-6):
            w(f"\n  Climate control (v10.8):\n")
            w(f"    E_slow:     mean={np.mean(clm_E_slow):.4f}  max={np.max(clm_E_slow):.4f}\n")
            w(f"    lr_meta:    mean={np.mean(clm_lr_meta):.4f}  min={np.min(clm_lr_meta):.4f}\n")
            w(f"    hold:       {np.sum(clm_hold)}/{total} ({np.mean(clm_hold):.1%})\n")
            if len(clm_eff) > 0:
                w(f"    lr_eff:     mean={np.mean(clm_eff):.6f}  min={np.min(clm_eff):.6f}\n")
        elif len(clm_lr_meta) > 0:
            w(f"\n  Climate: idle (lr_meta=1.0, E_slow~1.0)\n")

    # ── DS diagnostics ──
    w(f"\n{'═' * 80}\n")
    w(f"  DiversitySentinel Lambda PID Diagnostics\n")
    w(f"{'═' * 80}\n")

    for mode in ["div", "grav_div"]:
        if mode not in results:
            continue
        h = results[mode]["hist"]
        lam_raw = np.array(h.get("div_lambda_raw", []))
        lam_gov = np.array(h.get("div_lambda", []))
        lam_eff = np.array(h.get("div_lambda_eff", []))
        spread = np.array(h.get("div_spread", []))
        total = len(lam_gov)
        if total == 0:
            continue

        at_max = np.sum(lam_gov >= 0.499) / total
        at_min = np.sum(lam_gov <= 0.001) / total

        w(f"\n  {mode} ({total} steps):\n")
        w(f"    lambda_raw:  mean={np.mean(lam_raw):.4f}  min={np.min(lam_raw):.4f}  max={np.max(lam_raw):.4f}\n")
        w(f"    lambda_gov:  mean={np.mean(lam_gov):.4f}  min={np.min(lam_gov):.4f}  max={np.max(lam_gov):.4f}\n")
        if len(lam_eff) > 0:
            w(f"    lambda_eff:  mean={np.mean(lam_eff):.4f}  min={np.min(lam_eff):.4f}  max={np.max(lam_eff):.4f}\n")
        w(f"    spread:      mean={np.mean(spread):.4f}  min={np.min(spread):.4f}  max={np.max(spread):.4f}\n")
        w(f"    at_max (>=0.499): {at_max:.1%}\n")
        w(f"    at_min (<=0.001): {at_min:.1%}\n")

        n = min(steps_per_epoch, total)
        w(f"    last epoch: raw={np.mean(lam_raw[-n:]):.4f}  gov={np.mean(lam_gov[-n:]):.4f}\n")

        dl = h.get("grav_distress_level", [])
        if dl:
            dl = np.array(dl)
            w(f"    distress: L0={np.mean(dl==0):.1%}  L1(trip)={np.mean(dl==1):.1%}  L2(emerg)={np.mean(dl==2):.1%}\n")

    return buf.getvalue()

# ── Main sweep loop ──
all_results = {}  # {lr: {mode: results}}
all_summaries = {}  # {lr_str: {mode: metrics}}

for lr_val in LR_GRID:
    lr_str = f"{lr_val:.0e}"
    print(f"\n{'#' * 70}")
    print(f"  LR = {lr_val}  ({lr_str})")
    print(f"{'#' * 70}")

    # Override globals
    globals()['LR_G'] = lr_val
    globals()['LR_D'] = lr_val

    results = {}
    metrics_for_lr = {}

    for mode in MODES:
        print(f"\n  [{lr_str}] {mode}...", end=" ", flush=True)
        t0 = time.time()
        res = train_gan(mode=mode)
        elapsed = time.time() - t0
        results[mode] = res

        m = compute_all_metrics(res['G'], res['fake'])
        m['train_time'] = res['train_time']
        m['mode'] = mode

        h = res['hist']
        n_tail = steps_per_epoch * 5
        m['loss_D_final'] = float(np.mean(h['loss_D'][-n_tail:]))
        m['loss_G_final'] = float(np.mean(h['loss_G'][-n_tail:]))

        if 'grav' in mode:
            for k in ['calm', 'wall_any', 'comfort_active', 'i_enabled', 'panic_active']:
                m[f'grav_{k}_rate'] = float(np.mean(h[f'grav_{k}'][-n_tail:]))
            for k in ['wall_ema_fast', 'i_sat', 'gaps_db_nz', 'headroom_min', 'noise_norm',
                       'authority_factor', 'panic_factor', 'panic_shock_stress', 'panic_shock_stress_ema', 'noise_ref', 'u_cap_eff',
                       'auth_min_eff', 'intervention_rate', 'health_proxy',
                       'dyn_deadband_z_eff', 'world_drift', 'idle_fraction',
                       'climate_E_slow', 'climate_lr_meta', 'climate_hold_applied', 'climate_lr_eff_mean']:
                m[f'grav_{k}_mean'] = float(np.mean(h[f'grav_{k}'][-n_tail:]))
        if 'div' in mode:
            m['div_lambda_final'] = float(np.mean(h['div_lambda'][-n_tail:]))
            m['div_spread_final'] = float(np.mean(h['div_spread'][-n_tail:]))

        metrics_for_lr[mode] = m

        # Free G model + CUDA after metrics
        if 'G' in res:
            del res['G']
        import gc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        if IS_TOY:
            mc = m.get('n_modes_covered', '?')
            mt = m.get('n_modes_total', '?')
            print(f"modes={mc}/{mt}  KL={m.get('mode_kl',0):.3f}  ({elapsed:.0f}s)")
        else:
            fid_s = f"FID={m['fid']:.1f}" if 'fid' in m else "FID=?"
            kid_s = f"KID={m['kid_mean']:.4f}" if 'kid_mean' in m else "KID=?"
            print(f"{fid_s}  {kid_s}  ({elapsed:.0f}s)")

    all_results[lr_str] = results
    metrics_for_lr['_config'] = {
        'lr_G': lr_val, 'lr_D': lr_val, 'epochs': EPOCHS,
        'batch_size': BATCH_SIZE, 'seed': SEED, 'grav_version': GRAV_VERSION,
        'dataset': DATASET, 'steps_per_epoch': steps_per_epoch,
    }
    all_summaries[lr_str] = metrics_for_lr

# ── Save combined JSON ──
jp = os.path.join(OUTPUT_DIR, 'sweep_metrics.json')
with open(jp, 'w') as f:
    json.dump(all_summaries, f, indent=2, default=str)
print(f"\nSweep metrics saved: {jp}")

# ── Save combined diagnostic report ──
full_report = io.StringIO()
full_report.write(f"{'=' * 80}\n")
full_report.write(f"  LR SWEEP REPORT — GravBalancer {GRAV_VERSION}\n")
full_report.write(f"  Grid: {[f'{lr:.0e}' for lr in LR_GRID]}\n")
full_report.write(f"  Epochs={EPOCHS}  Batch={BATCH_SIZE}  Seed={SEED}  Grid={GRID_SIZE}x{GRID_SIZE}\n")
full_report.write(f"{'=' * 80}\n\n")

# ── Compact cross-LR summary table ──
full_report.write(f"{'=' * 80}\n")
full_report.write(f"  CROSS-LR SUMMARY\n")
full_report.write(f"{'=' * 80}\n")
full_report.write(f"{'LR':>8s} | {'baseline':>22s} | {'grav':>22s} | {'div':>22s} | {'grav_div':>22s}\n")
full_report.write(f"{'':>8s} | {'modes  CV    KL':>22s} | {'modes  CV    KL':>22s} | {'modes  CV    KL':>22s} | {'modes  CV    KL':>22s}\n")
full_report.write(f"{'-' * 108}\n")
for lr_str, mets in all_summaries.items():
    parts = [f"{lr_str:>8s}"]
    for mode in ["baseline", "grav", "div", "grav_div"]:
        if mode in mets:
            m = mets[mode]
            mc = m.get('n_modes_covered', 0)
            cv = m.get('uniformity_cv', 0)
            kl = m.get('mode_kl', 0)
            parts.append(f"{mc:>4d}  {cv:.3f}  {kl:.3f}")
        else:
            parts.append(f"{'—':>22s}")
    full_report.write(" | ".join(parts) + "\n")
full_report.write("\n")

# ── Grav-specific cross-LR table ──
full_report.write(f"{'=' * 80}\n")
full_report.write(f"  GRAV INTERNALS ACROSS LR\n")
full_report.write(f"{'=' * 80}\n")
full_report.write(f"{'LR':>8s} {'mode':>10s} | {'calm%':>7s} {'trip%':>7s} {'auth':>7s} {'stress':>7s}"
                  f" {'panic%':>7s} {'noise_n':>7s} {'wall%':>7s} {'I_en%':>7s}"
                  f" {'db_eff':>7s} {'idle%':>7s} {'interv':>7s} {'amin':>7s}"
                  f" {'lr_meta':>7s} {'hold%':>7s}\n")
full_report.write(f"{'-' * 120}\n")
for lr_str, mets in all_summaries.items():
    for mode in ["grav", "grav_div"]:
        if mode not in mets:
            continue
        res = all_results[lr_str][mode]
        h = res["hist"]
        calm_r = np.mean(h.get("grav_calm", [0]))*100
        trip_r = np.mean(h.get("grav_trip_active", [0]))*100
        auth = np.mean(h.get("grav_authority_factor", [0]))
        stress = np.mean(h.get("grav_control_stress", [0]))
        panic = np.mean(h.get("grav_panic_active", [0]))*100
        nn = np.mean(h.get("grav_noise_norm", [0]))
        wall = np.mean(h.get("grav_wall_any", [0]))*100
        i_en = np.mean(h.get("grav_i_enabled", [0]))*100
        db_eff = np.mean(h.get("grav_dyn_deadband_z_eff", [0]))
        idle_f = np.mean(h.get("grav_idle_fraction", [0]))*100
        interv = np.mean(h.get("grav_intervention_rate", [0]))*100
        amin = np.mean(h.get("grav_auth_min_eff", [0]))
        clm_meta = np.mean(h.get("grav_climate_lr_meta", [1.0]))
        clm_hold = np.mean(h.get("grav_climate_hold_applied", [0]))*100
        full_report.write(f"{lr_str:>8s} {mode:>10s} | {calm_r:>6.1f}% {trip_r:>6.1f}% {auth:>7.3f} {stress:>7.3f}"
                          f" {panic:>6.1f}% {nn:>7.3f} {wall:>6.1f}% {i_en:>6.1f}%"
                          f" {db_eff:>7.3f} {idle_f:>6.1f}% {interv:>6.1f}% {amin:>7.4f}"
                          f" {clm_meta:>7.3f} {clm_hold:>6.1f}%\n")
full_report.write("\n")

# ── Per-LR detailed reports ──
for lr_str in all_summaries:
    lr_val = float(all_summaries[lr_str]['_config']['lr_G'])
    config_vars = {
        'dataset': DATASET, 'grid_size': GRID_SIZE, 'grid_spacing': GRID_SPACING,
        'grid_std': GRID_STD, 'n_data': N_DATA, 'epochs': EPOCHS,
        'lr_G': lr_val, 'lr_D': lr_val, 'batch_size': BATCH_SIZE,
        'z_dim': Z_DIM, 'seed': SEED, 'steps_per_epoch': steps_per_epoch,
        'total_steps': EPOCHS * steps_per_epoch,
        'grav_version': GRAV_VERSION, 'grav_warmup_epochs': GRAV_WARMUP_EPOCHS,
        'div_lambda_max': DIV_LAMBDA_MAX, 'div_pid_target_ratio': DIV_PID_TARGET_RATIO,
    }
    lr_report = generate_report(all_results[lr_str], all_summaries[lr_str], config_vars)
    full_report.write(lr_report)
    full_report.write("\n\n")

report_text = full_report.getvalue()
rp = os.path.join(OUTPUT_DIR, 'sweep_report.txt')
with open(rp, 'w', encoding='utf-8') as f:
    f.write(report_text)
print(f"Sweep report saved: {rp} ({len(report_text)} chars)")

# ── Print compact summary ──
print(f"\n{'=' * 80}")
print(f"  SWEEP COMPLETE")
print(f"{'=' * 80}")
for lr_str, mets in all_summaries.items():
    line = f"  LR={lr_str:>8s}:"
    for mode in ["baseline", "grav", "div", "grav_div"]:
        if mode in mets:
            if IS_TOY:
                mc = mets[mode].get('n_modes_covered', 0)
                kl = mets[mode].get('mode_kl', 0)
                line += f"  {mode}={mc:>3d}m/{kl:.2f}kl"
            else:
                fid = mets[mode].get('fid', -1)
                kid = mets[mode].get('kid_mean', -1)
                fid_s = f"{fid:.0f}" if fid >= 0 else "?"
                kid_s = f"{kid:.3f}" if kid >= 0 else "?"
                line += f"  {mode}=F{fid_s}/K{kid_s}"
    print(line)




######################################################################
  LR = 0.005  (5e-03)
######################################################################

  [5e-03] baseline...   [baseline] ep   1/300  D=2.806  G=1.787
  [baseline] ep  10/300  D=1.829  G=0.762
  [baseline] ep  20/300  D=1.404  G=1.309
  [baseline] ep  30/300  D=0.911  G=1.913
  [baseline] ep  40/300  D=1.697  G=0.801
  [baseline] ep  50/300  D=1.519  G=1.011
  [baseline] ep  60/300  D=1.384  G=1.036
  [baseline] ep  70/300  D=0.963  G=1.784
  [baseline] ep  80/300  D=0.864  G=2.186
  [baseline] ep  90/300  D=0.633  G=2.499
  [baseline] ep 100/300  D=0.658  G=3.120
  [baseline] ep 110/300  D=0.002  G=4.834
  [baseline] ep 120/300  D=0.319  G=3.382
  [baseline] ep 130/300  D=0.011  G=5.317
  [baseline] ep 140/300  D=0.540  G=3.627
  [baseline] ep 150/300  D=0.271  G=3.176
  [baseline] ep 160/300  D=0.110  G=3.296
  [baseline] ep 170/300  D=0.287  G=3.297
  [baseline] ep 180/300  D=0.426  G=3.140
  [baseline] e

C:\Conda\lib\site-packages\torchmetrics\utilities\prints.py:36: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


FID=135.4  KID=0.0902  (6554s)

  [5e-03] grav...   [grav] ep   1/300  D=2.839  G=2.680 [warmup]
  [grav] ep  10/300  D=1.913  G=0.530 calm=0% trip=0% [none] auth=0.341 stress=0.001 db=3.50 idle=96%
  [grav] ep  20/300  D=1.728  G=0.871 calm=0% trip=0% [none] auth=0.283 stress=0.000 db=3.50 idle=100% lr_meta=0.973 hold=0%
  [grav] ep  30/300  D=1.542  G=1.077 calm=0% trip=8% [none] auth=0.231 stress=0.001 db=3.50 idle=100% lr_meta=0.959 hold=0%
  [grav] ep  40/300  D=0.873  G=2.126 calm=0% trip=0% [none] auth=0.318 stress=0.000 db=3.50 idle=100% lr_meta=0.967 hold=0%
  [grav] ep  50/300  D=0.956  G=2.165 calm=0% trip=0% [none] auth=0.351 stress=0.000 db=3.50 idle=100% lr_meta=0.975 hold=0%
  [grav] ep  60/300  D=0.658  G=2.685 calm=0% trip=0% [none] auth=0.309 stress=0.003 db=3.50 idle=100% lr_meta=0.980 hold=0%
  [grav] ep  70/300  D=0.729  G=2.889 calm=0% trip=0% [none] auth=0.313 stress=0.021 db=3.50 idle=100% lr_meta=0.976 hold=0%
  [grav] ep  80/300  D=0.593  G=3.222 calm=0% trip=